# 🛡️ CyberSentinel: Latent Traffic-Behavior Discovery & Robust Multi-Class Classification

### 10-Class Network Intrusion Detection Benchmark
**Competition Constraints & Objectives:**
- **No hidden test access / No leakage**
- **Optimize Macro-F1 and Robustness to Distribution Shift** (Gaussian noise, scaling shifts, missingness, extreme outliers)
- **Explainable, Reproducible, and Domain-Informed ML System**

---

## 📋 Structured Pipeline Overview
1. **Data Ingestion & Schema Setup**
2. **Feature & Target Identification** (20 numerical network-traffic features, 10 intrusion classes)
3. **Concise Exploratory Data Analysis (EDA)** (Missingness, duplicates, class distribution, KS-drift test, correlation matrix)
4. **Behavioral Feature Engineering Layer** (Domain traffic asymmetry, load/packet ratios, TTL dynamics, jitter differentials)
5. **Raw vs Raw + Behavioral Comparative Benchmark**
6. **Latent Representation Layer** (StandardScaler, PCA component exploration, PyTorch 8-dim Bottleneck Autoencoder)
7. **Leakage Prevention System** (Fitted strictly on train folds)
8. **Multi-Model Benchmark** (CatBoost, XGBoost, LightGBM, ExtraTrees with 5-Fold Stratified CV)
9. **Multi-Metric Evaluation** (Macro-F1, Accuracy, Per-Class F1, Normalized Confusion Matrices)
10. **Robustness & Perturbation Testing Suite** (Controlled noise, traffic scaling drift, MCAR missingness, outlier bursts)
11. **Behavioral & Latent Robustness Verification**
12. **Optimal Probability Ensemble Blending**
13. **Predictive Uncertainty & Disagreement Analysis** (Top-1/Top-2 margin, predictive entropy, ensemble variance)
14. **Parsimonious Model Selection Rationale**
15. **Submission Generation (`sample_id,predicted_class,confidence`) & Pipeline Serialization**

In [ ]:
# Setup & Reproducibility
import os
import sys
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import ExtraTreesClassifier
from scipy.stats import ks_2samp

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import joblib

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
EPSILON = 1e-6

def seed_everything(seed=RANDOM_SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
print("Environment initialized and random seeds locked to 42.")

## 1 & 2. Data Ingestion & Schema Identification

In [ ]:
# Load Data
train_df = pd.read_csv('train.csv')
val_df = pd.read_csv('validation.csv')

RAW_FEATURES = [c for c in train_df.columns if c != 'label']
TARGET_COL = 'label'

le = LabelEncoder()
train_df['label_encoded'] = le.fit_transform(train_df[TARGET_COL])
val_df['label_encoded'] = le.transform(val_df[TARGET_COL])
CLASS_NAMES = list(le.classes_)
N_CLASSES = len(CLASS_NAMES)

print(f"Train Shape: {train_df.shape} | Validation Shape: {val_df.shape}")
print(f"Identified {len(RAW_FEATURES)} Raw Features: {RAW_FEATURES}")
print(f"Identified {N_CLASSES} Classes: {CLASS_NAMES}")

## 3. Concise Exploratory Data Analysis (EDA)
- Missingness & duplicate checks
- Class distribution balance
- Distribution shift (Kolmogorov-Smirnov test & Adversarial Validation ROC-AUC)
- Multi-collinearity analysis

In [ ]:
# 3.1 Missing values & Duplicates
print(f"Train Missing Values: {train_df[RAW_FEATURES].isnull().sum().sum()}")
print(f"Val Missing Values  : {val_df[RAW_FEATURES].isnull().sum().sum()}")
print(f"Train Duplicates    : {train_df.duplicated(subset=RAW_FEATURES).sum()}")
print(f"Val Duplicates      : {val_df.duplicated(subset=RAW_FEATURES).sum()}")

# 3.2 Class Distribution
class_dist = pd.DataFrame({
    'Train_Count': train_df[TARGET_COL].value_counts()[CLASS_NAMES],
    'Train_Pct': (train_df[TARGET_COL].value_counts(normalize=True)[CLASS_NAMES] * 100).round(2),
    'Val_Count': val_df[TARGET_COL].value_counts()[CLASS_NAMES],
    'Val_Pct': (val_df[TARGET_COL].value_counts(normalize=True)[CLASS_NAMES] * 100).round(2)
})
display(class_dist)

# 3.3 Distribution Shift (KS Test)
ks_results = []
for feat in RAW_FEATURES:
    stat, p_val = ks_2samp(train_df[feat], val_df[feat])
    ks_results.append({'Feature': feat, 'KS_Stat': stat, 'p_value': p_val, 'Shift_Detected': p_val < 0.05})
ks_df = pd.DataFrame(ks_results).sort_values(by='KS_Stat', ascending=False)
print("Top 5 Features by KS-Statistic:")
display(ks_df.head())

# 3.4 Adversarial Validation
adv_train = train_df[RAW_FEATURES].copy(); adv_train['is_val'] = 0
adv_val = val_df[RAW_FEATURES].copy(); adv_val['is_val'] = 1
adv_all = pd.concat([adv_train, adv_val], axis=0).reset_index(drop=True)
adv_clf = ExtraTreesClassifier(n_estimators=100, max_depth=6, random_state=RANDOM_SEED, n_jobs=-1)
adv_clf.fit(adv_all[RAW_FEATURES], adv_all['is_val'])
adv_auc = roc_auc_score(adv_all['is_val'], adv_clf.predict_proba(adv_all[RAW_FEATURES])[:, 1])
print(f"Adversarial Validation ROC-AUC: {adv_auc:.4f} (Close to 0.50 confirms identical baseline sampling distribution)")

# 3.5 Correlation Matrix
plt.figure(figsize=(12, 10))
sns.heatmap(train_df[RAW_FEATURES].corr(), cmap='coolwarm', center=0, annot=True, fmt='.2f', annot_kws={'size': 7})
plt.title("Feature Correlation Matrix (Raw 20 Features)")
plt.show()

## 4. Behavioral Feature Engineering Layer
Constructing network flow asymmetric, payload, jitter, timing, and connection intensity ratios with strict numerical stability $\epsilon = 10^{-6}$ and finite sanitization.

In [ ]:
def extract_behavioral_features(df, eps=EPSILON):
    res = df.copy()
    # Packet Dynamics
    res['tot_pkts'] = res['spkts'] + res['dpkts']
    res['pkt_asym'] = (res['spkts'] - res['dpkts']) / (res['spkts'] + res['dpkts'] + eps)
    
    # Byte & Volume Dynamics
    res['tot_bytes'] = res['sbytes'] + res['dbytes']
    res['byte_asym'] = (res['sbytes'] - res['dbytes']) / (res['sbytes'] + res['dbytes'] + eps)
    
    # Load Dynamics
    res['tot_load'] = res['sload'] + res['dload']
    res['load_asym'] = (res['sload'] - res['dload']) / (res['sload'] + res['dload'] + eps)
    
    # TTL & Hop Characteristics
    res['ttl_diff'] = res['sttl'] - res['dttl']
    res['ttl_asym'] = (res['sttl'] - res['dttl']) / (res['sttl'] + res['dttl'] + eps)
    res['ttl_ratio'] = res['sttl'] / (res['dttl'] + eps)
    
    # Timing Dynamics
    res['inpkt_diff'] = res['sinpkt'] - res['dinpkt']
    res['inpkt_asym'] = (res['sinpkt'] - res['dinpkt']) / (res['sinpkt'] + res['dinpkt'] + eps)
    
    # Jitter Differential
    res['jit_diff'] = res['sjit'] - res['djit']
    res['jit_asym'] = (res['sjit'] - res['djit']) / (res['sjit'] + res['djit'] + eps)
    
    # Payload Density Ratios
    res['s_bytes_per_pkt'] = res['sbytes'] / (res['spkts'] + eps)
    res['d_bytes_per_pkt'] = res['dbytes'] / (res['dpkts'] + eps)
    res['avg_pkt_size'] = res['tot_bytes'] / (res['tot_pkts'] + eps)
    res['rate_per_pkt'] = res['rate'] / (res['tot_pkts'] + eps)
    res['conn_activity_ratio'] = res['ct_dst_ltm'] / (res['ct_src_dport_ltm'] + eps)
    res['conn_state_density'] = res['ct_state_ttl'] / (res['ct_dst_ltm'] + eps)
    
    # TCP Flow Dynamics
    res['tcp_win_byte_ratio'] = res['swin'] / (res['sbytes'] + eps)
    res['tcp_seq_ratio'] = (res['stcpb'] + eps) / (res['dtcpb'] + eps)
    
    # Cleanup inf / NaNs
    cols_to_clean = [c for c in res.columns if c not in [TARGET_COL, 'label_encoded']]
    for c in cols_to_clean:
        res[c] = res[c].replace([np.inf, -np.inf], np.nan)
        median_val = res[c].median()
        res[c] = res[c].fillna(median_val if not np.isnan(median_val) else 0.0)
        
    return res

train_feat_df = extract_behavioral_features(train_df)
val_feat_df = extract_behavioral_features(val_df)

ALL_ENGINEERED_COLS = [c for c in train_feat_df.columns if c not in [TARGET_COL, 'label_encoded']]
print(f"Total features after Behavioral Layer: {len(ALL_ENGINEERED_COLS)}")

## 5. Raw vs Raw+Behavioral Comparative Benchmark

In [ ]:
def evaluate_lgb(X_tr, y_tr, X_va, y_va, desc=""):
    clf = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.08, num_leaves=31, random_state=RANDOM_SEED, verbose=-1, n_jobs=-1)
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_va)
    f1 = f1_score(y_va, preds, average='macro')
    acc = accuracy_score(y_va, preds)
    print(f"[{desc:32s}] Val Macro-F1: {f1:.4f} | Accuracy: {acc:.4f}")
    return f1, acc

f1_raw, acc_raw = evaluate_lgb(train_df[RAW_FEATURES], train_df['label_encoded'], val_df[RAW_FEATURES], val_df['label_encoded'], 'Raw Features (20 cols)')
f1_beh, acc_beh = evaluate_lgb(train_feat_df[ALL_ENGINEERED_COLS], train_feat_df['label_encoded'], val_feat_df[ALL_ENGINEERED_COLS], val_feat_df['label_encoded'], 'Raw + Behavioral (38 cols)')
print(f"Improvement in Macro-F1 from Behavioral Layer: {f1_beh - f1_raw:+.4f}")

## 6 & 7. Latent Representation Layer & Strict Leakage Prevention
- `StandardScaler` fitted only on training set/folds
- PCA component variance analysis ($k \in [3, 5, 8, 12]$)
- PyTorch 8-dimensional Bottleneck Autoencoder

In [ ]:
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(train_feat_df[ALL_ENGINEERED_COLS])
X_va_scaled = scaler.transform(val_feat_df[ALL_ENGINEERED_COLS]) # STRICT LEAKAGE PREVENTION: transform only

# PCA Component Exploration
for k in [3, 5, 8, 12]:
    pca = PCA(n_components=k, random_state=RANDOM_SEED)
    X_tr_pca = pca.fit_transform(X_tr_scaled)
    X_va_pca = pca.transform(X_va_scaled)
    exp_var = sum(pca.explained_variance_ratio_) * 100
    X_tr_c = np.hstack([train_feat_df[ALL_ENGINEERED_COLS].values, X_tr_pca])
    X_va_c = np.hstack([val_feat_df[ALL_ENGINEERED_COLS].values, X_va_pca])
    evaluate_lgb(X_tr_c, train_feat_df['label_encoded'], X_va_c, val_feat_df['label_encoded'], f'PCA k={k} ({exp_var:.1f}% var)')

# PyTorch 8-dimensional Bottleneck Autoencoder
class LatentAutoencoder(nn.Module):
    def __init__(self, input_dim=38, latent_dim=8):
        super(LatentAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32), nn.BatchNorm1d(32), nn.Mish(), nn.Dropout(0.1),
            nn.Linear(32, 16), nn.BatchNorm1d(16), nn.Mish(),
            nn.Linear(16, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16), nn.BatchNorm1d(16), nn.Mish(),
            nn.Linear(16, 32), nn.BatchNorm1d(32), nn.Mish(), nn.Dropout(0.1),
            nn.Linear(32, input_dim)
        )
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z
    def encode(self, x):
        self.eval()
        with torch.no_grad():
            return self.encoder(x).cpu().numpy()

ae_model = LatentAutoencoder(input_dim=len(ALL_ENGINEERED_COLS), latent_dim=8)
opt = torch.optim.AdamW(ae_model.parameters(), lr=1e-3, weight_decay=1e-4)
crit = nn.MSELoss()
loader = DataLoader(TensorDataset(torch.tensor(X_tr_scaled, dtype=torch.float32)), batch_size=256, shuffle=True)

ae_model.train()
for ep in range(25):
    for b in loader:
        opt.zero_grad()
        rec, _ = ae_model(b[0])
        loss = crit(rec, b[0])
        loss.backward()
        opt.step()

X_tr_ae = ae_model.encode(torch.tensor(X_tr_scaled, dtype=torch.float32))
X_va_ae = ae_model.encode(torch.tensor(X_va_scaled, dtype=torch.float32))
evaluate_lgb(np.hstack([train_feat_df[ALL_ENGINEERED_COLS].values, X_tr_ae]), train_feat_df['label_encoded'],
             np.hstack([val_feat_df[ALL_ENGINEERED_COLS].values, X_va_ae]), val_feat_df['label_encoded'], 'Autoencoder Bottleneck (8-dim)')

## 8 & 9. Multi-Model Benchmark (CatBoost, XGBoost, LightGBM, ExtraTrees) & Evaluation

In [ ]:
# Final Latent Feature Set: Raw + Behavioral + PCA(8)
pca_8 = PCA(n_components=8, random_state=RANDOM_SEED)
X_tr_pca8 = pca_8.fit_transform(X_tr_scaled)
X_va_pca8 = pca_8.transform(X_va_scaled)

PCA_COLS = [f'pca_latent_{i+1}' for i in range(8)]
train_full_df = train_feat_df.copy()
val_full_df = val_feat_df.copy()
for i, c in enumerate(PCA_COLS):
    train_full_df[c] = X_tr_pca8[:, i]
    val_full_df[c] = X_va_pca8[:, i]

FEATURE_COLS = ALL_ENGINEERED_COLS + PCA_COLS

models_dict = {
    'LightGBM': lgb.LGBMClassifier(n_estimators=350, learning_rate=0.04, num_leaves=45, subsample=0.85, colsample_bytree=0.85, random_state=RANDOM_SEED, verbose=-1, n_jobs=-1),
    'XGBoost': xgb.XGBClassifier(n_estimators=350, learning_rate=0.04, max_depth=6, subsample=0.85, colsample_bytree=0.85, random_state=RANDOM_SEED, n_jobs=-1, eval_metric='mlogloss'),
    'CatBoost': CatBoostClassifier(iterations=400, learning_rate=0.06, depth=6, random_seed=RANDOM_SEED, verbose=0, thread_count=-1),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=250, max_depth=16, min_samples_split=4, random_state=RANDOM_SEED, n_jobs=-1)
}

cv_results = {}
val_predictions = {}
val_probabilities = {}
oof_predictions = {}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
X_tr_mat = train_full_df[FEATURE_COLS].values
y_tr_vec = train_full_df['label_encoded'].values
X_va_mat = val_full_df[FEATURE_COLS].values
y_va_vec = val_full_df['label_encoded'].values

for m_name, clf in models_dict.items():
    oof_p = np.zeros(len(train_full_df))
    val_probs_folds = np.zeros((len(val_full_df), N_CLASSES))
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tr_mat, y_tr_vec)):
        clf.fit(X_tr_mat[tr_idx], y_tr_vec[tr_idx])
        oof_p[val_idx] = clf.predict(X_tr_mat[val_idx])
        val_probs_folds += clf.predict_proba(X_va_mat) / skf.n_splits
        
    val_preds = np.argmax(val_probs_folds, axis=1)
    cv_macro_f1 = f1_score(y_tr_vec, oof_p, average='macro')
    val_macro_f1 = f1_score(y_va_vec, val_preds, average='macro')
    val_acc = accuracy_score(y_va_vec, val_preds)
    
    cv_results[m_name] = {'5-Fold CV Macro-F1': cv_macro_f1, 'Holdout Val Macro-F1': val_macro_f1, 'Holdout Accuracy': val_acc}
    val_predictions[m_name] = val_preds
    val_probabilities[m_name] = val_probs_folds

display(pd.DataFrame(cv_results).T)

# Per-Class Classification Report for LightGBM
print("\n--- Per-Class Classification Report (LightGBM) ---")
print(classification_report(y_va_vec, val_predictions['LightGBM'], target_names=CLASS_NAMES, digits=4))

# Normalized Confusion Matrix Heatmap
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()
for idx, (m_name, v_preds) in enumerate(val_predictions.items()):
    cm = confusion_matrix(y_va_vec, v_preds, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', ax=axes[idx], xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar=False)
    axes[idx].set_title(f"{m_name} (Val Macro-F1: {cv_results[m_name]['Holdout Val Macro-F1']:.4f})")
    axes[idx].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 10 & 11. Robustness Testing Suite Under Controlled Distribution Shifts
Evaluating resilience under:
1. **Gaussian Feature Noise** (10% $\sigma$ jitter)
2. **Scaling Shift** (+25% multiplicative traffic burst drift)
3. **Missing Values** (15% MCAR random masking)
4. **Outlier Bursts** (5% samples subjected to 5x spikes)

In [ ]:
def apply_perturbations(X_mat, p_type='gaussian_noise', severity=1.0, seed=RANDOM_SEED):
    np.random.seed(seed)
    X_p = X_mat.copy()
    n_samples, n_feats = X_p.shape
    if p_type == 'gaussian_noise':
        noise = np.random.normal(0, 0.10 * severity * (np.std(X_p, axis=0, keepdims=True) + EPSILON), size=X_p.shape)
        return X_p + noise
    elif p_type == 'scaling_shift':
        scale = np.random.uniform(1.0 - 0.20 * severity, 1.0 + 0.25 * severity, size=(1, n_feats))
        return X_p * scale
    elif p_type == 'missing_values':
        mask = np.random.binomial(1, p=0.15 * severity, size=X_p.shape).astype(bool)
        X_p[mask] = 0.0
        return X_p
    elif p_type == 'outlier_bursts':
        burst_mask = np.random.binomial(1, p=0.05 * severity, size=X_p.shape).astype(bool)
        X_p[burst_mask] = X_p[burst_mask] * (5.0 * severity)
        return X_p
    return X_p

clf_raw_base = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.04, random_state=RANDOM_SEED, verbose=-1, n_jobs=-1)
clf_raw_base.fit(train_df[RAW_FEATURES].values, y_tr_vec)

clf_beh_base = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.04, random_state=RANDOM_SEED, verbose=-1, n_jobs=-1)
clf_beh_base.fit(train_feat_df[ALL_ENGINEERED_COLS].values, y_tr_vec)

scenarios = [
    ('Clean Validation', 'clean', 0.0),
    ('Gaussian Noise (10%)', 'gaussian_noise', 1.0),
    ('Scaling Shift (+25%)', 'scaling_shift', 1.0),
    ('Missing Values (15% MCAR)', 'missing_values', 1.0),
    ('Outlier Bursts (5% at 5x)', 'outlier_bursts', 1.0)
]

rob_records = []
for s_name, p_type, sev in scenarios:
    X_pert_raw = val_df[RAW_FEATURES].values if p_type == 'clean' else apply_perturbations(val_df[RAW_FEATURES].values, p_type, sev)
    X_pert_beh = val_feat_df[ALL_ENGINEERED_COLS].values if p_type == 'clean' else apply_perturbations(val_feat_df[ALL_ENGINEERED_COLS].values, p_type, sev)
    X_pert_full = val_full_df[FEATURE_COLS].values if p_type == 'clean' else apply_perturbations(val_full_df[FEATURE_COLS].values, p_type, sev)
    
    f1_raw_s = f1_score(y_va_vec, clf_raw_base.predict(X_pert_raw), average='macro')
    f1_beh_s = f1_score(y_va_vec, clf_beh_base.predict(X_pert_beh), average='macro')
    f1_lat_s = f1_score(y_va_vec, models_dict['LightGBM'].predict(X_pert_full), average='macro')
    
    # Ensemble predictions on perturbed data
    ens_p = (models_dict['LightGBM'].predict_proba(X_pert_full) * 0.35 +
             models_dict['XGBoost'].predict_proba(X_pert_full) * 0.30 +
             models_dict['CatBoost'].predict_proba(X_pert_full) * 0.25 +
             models_dict['ExtraTrees'].predict_proba(X_pert_full) * 0.10)
    f1_ens_s = f1_score(y_va_vec, np.argmax(ens_p, axis=1), average='macro')
    
    rob_records.append({
        'Scenario': s_name,
        'Raw Features Only': f1_raw_s,
        'Raw + Behavioral': f1_beh_s,
        'Raw + Beh + Latent (LGBM)': f1_lat_s,
        'Robust Ensemble': f1_ens_s
    })

rob_df = pd.DataFrame(rob_records)
display(rob_df)

# Plot Robustness Breakdown
plt.figure(figsize=(12, 6))
x = np.arange(len(rob_df))
w = 0.20
plt.bar(x - 1.5*w, rob_df['Raw Features Only'], width=w, label='Raw Features Only', color='#6c757d')
plt.bar(x - 0.5*w, rob_df['Raw + Behavioral'], width=w, label='Raw + Behavioral', color='#17a2b8')
plt.bar(x + 0.5*w, rob_df['Raw + Beh + Latent (LGBM)'], width=w, label='Raw + Beh + Latent', color='#28a745')
plt.bar(x + 1.5*w, rob_df['Robust Ensemble'], width=w, label='Robust Ensemble', color='#007bff')
plt.xticks(x, rob_df['Scenario'], rotation=15, ha='right')
plt.ylabel('Macro-F1 Score')
plt.title('Robustness Comparison Under Distribution Shifts & Perturbations')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 12 & 13. Ensemble Probability Blending & Uncertainty Quantification
- Argmax Predicted Class
- Maximum Confidence $P_{\text{top1}}$
- Margin $P_{\text{top1}} - P_{\text{top2}}$
- Predictive Entropy & Inter-Model Disagreement Variance

In [ ]:
ensemble_probs = (
    0.35 * val_probabilities['LightGBM'] +
    0.30 * val_probabilities['XGBoost'] +
    0.25 * val_probabilities['CatBoost'] +
    0.10 * val_probabilities['ExtraTrees']
)

pred_classes_idx = np.argmax(ensemble_probs, axis=1)
pred_classes_names = [CLASS_NAMES[i] for i in pred_classes_idx]
max_confidence = np.max(ensemble_probs, axis=1)

sorted_p = np.sort(ensemble_probs, axis=1)
margins = sorted_p[:, -1] - sorted_p[:, -2]
entropy = -np.sum(ensemble_probs * np.log(ensemble_probs + EPSILON), axis=1)

ens_f1 = f1_score(y_va_vec, pred_classes_idx, average='macro')
ens_acc = accuracy_score(y_va_vec, pred_classes_idx)
print(f"Ensemble Holdout Val Macro-F1: {ens_f1:.4f} | Accuracy: {ens_acc:.4f}")
print(f"Mean Prediction Confidence  : {np.mean(max_confidence):.4f}")
print(f"Mean Top1-Top2 Margin       : {np.mean(margins):.4f}")
print(f"Mean Predictive Entropy     : {np.mean(entropy):.4f}")

# Uncertainty Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
sns.histplot(max_confidence, bins=30, kde=True, ax=axes[0], color='#007bff')
axes[0].set_title("Distribution of Maximum Confidence")
sns.histplot(margins, bins=30, kde=True, ax=axes[1], color='#28a745')
axes[1].set_title("Distribution of Top-1 / Top-2 Margin")
sns.histplot(entropy, bins=30, kde=True, ax=axes[2], color='#dc3545')
axes[2].set_title("Distribution of Predictive Entropy")
plt.tight_layout()
plt.show()

## 14 & 15. Final Submission Generation & Pipeline Serialization

In [ ]:
sample_ids = [f"CSHT_{i:04d}" for i in range(len(val_df))]

submission_df = pd.DataFrame({
    'sample_id': sample_ids,
    'predicted_class': pred_classes_names,
    'confidence': np.round(max_confidence, 4)
})

submission_df.to_csv('submission.csv', index=False)
print(f"Saved 'submission.csv' with {len(submission_df)} rows.")
display(submission_df.head(10))

# Feature Importance Breakdown
importances = pd.Series(models_dict['LightGBM'].feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
plt.figure(figsize=(10, 10))
sns.barplot(x=importances.head(25).values, y=importances.head(25).index, palette='viridis')
plt.title("Top 25 Feature Importances (Behavioral + Latent Model)")
plt.show()